In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from main import run_experiment, save_results
from simlm.config import (
    Config,
    ExperimentConfig,
    LLMConfig,
    GroundConfig,
)
import numpy as np

In [3]:
experiment_types = ["baseline_cot", "simlm"]
max_iterations = [3,5,7]
few_shots = [0, 1, 2]
models = [
    ("openai", "gpt-3.5-turbo"),
    ("openai", "gpt-4.1-nano"),
    ("google", "gemini-2.0-flash"),
    ("google", "gemini-2.5-flash-preview-04-17"),
    ("google", "gemini-1.5-flash"),
]
flat_grounds = [
    {"type": "flat"}
]
sine_grounds = [{"type": "sine", "amplitude": 0.5, "frequency": 1.0}]
interpolated_grounds = [
    {
        "type": "interpolated",
        "difficulty": 1.0,
        "easy": {"amplitude": 0.15, "frequency": 0.25},
        "hard": {
            "amplitudes": [0.6, 0.15, 0.05],
            "frequencies": [0.9, 2.25, 4.5],
        },
    }
    for difficulty in np.arange(0.0, 1.1, 0.1)
]
grounds = flat_grounds + sine_grounds + interpolated_grounds

In [4]:
# get the product of experiment types, models, and grounds with
# itertools.product
from itertools import product

configs = []

for experiment_type, (model_service, model_name), ground, max_iteration, few_shot in product(
    experiment_types,
    models,
    grounds,
    max_iterations,
    few_shots
    
):
    config = Config(
        experiment=ExperimentConfig(
            type=experiment_type,
            visualize=False,
            save_results=True,
            max_iterations=max_iteration,
            few_shot=few_shot,
            few_shot_examples_path="results\experiment_results-2520.jsonl"

        ),
        llm=LLMConfig(
            service=model_service,
            model_name=model_name,
            temperature=0.5,
        ),
        ground=GroundConfig(**ground),
    )
    configs.append(config)
    # print(config)
print(len(configs))

1170


In [5]:
configs = configs[-162:]

In [6]:
for config in configs:
    print(config)
    runner, results = run_experiment(config)
    save_results(results)

2025-04-29 22:31:48,962 - INFO - AFC is enabled with max remote calls: 10.


experiment=ExperimentConfig(type='simlm', visualize=False, save_results=True, few_shot_examples_path='results\\experiment_results-2520.jsonl', target_distance=50.0, target_bounce_number=3, tolerance=1, max_iterations=3, few_shot=0) llm=LLMConfig(service='google', model_name='gemini-2.5-flash-preview-04-17', temperature=0.5) simulation=SimulationConfig(fps=1000, max_duration=20, gravity=Coordinate(x=0.0, y=-9.81, z=0.0)) projectile=ProjectileConfig(elasticity=0.9, mass=1.0, radius=0.05) ground=GroundConfig(type='interpolated', friction=0.8, x_min=-200, x_max=400, step=0.1, amplitude=1, frequency=0.5, difficulty=1.0, easy=EasyGroundConfig(amplitude=0.15, frequency=0.25), hard=HardGroundConfig(amplitudes=[0.6, 0.15, 0.05], frequencies=[0.9, 2.25, 4.5]))

Running SimLM for InterpolatedGround(difficulty=1.0, easy_ground=EasyGround(amplitude=0.15, frequency=0.25), hard_ground=HardGround(amplitudes=[0.6  0.15 0.05], frequencies=[0.9  2.25 4.5 ]))

SimLM Iteration 1/3


2025-04-29 22:32:07,435 - INFO - AFC remote call 1 is done.
2025-04-29 22:32:07,436 - INFO - GoogleClient model:gemini-2.5-flash-preview-04-17, response: ```json
{
  "reasoning": "Initial estimate based on flat ground approximation: For 3 bounces to cover 50m with elasticity 0.9, the total horizontal distance is approximately v * sqrt(2h/g) * (1 + 2*elasticity + 2*elasticity^2). With g=9.81 and elasticity=0.9, this is v * sqrt(2h/9.81) * (1 + 1.8 + 1.62) = v * sqrt(0.2039h) * 4.42. Setting this to 50m gives v * sqrt(h) * sqrt(0.2039) * 4.42 ≈ 50, so v * sqrt(h) * 0.4515 * 4.42 ≈ 50, or v * sqrt(h) * 2.0 ≈ 50, leading to v * sqrt(h) ≈ 25.  A combination like h=6m and v=12m/s gives v*sqrt(h) = 12 * sqrt(6) ≈ 29.4, predicting a flat ground distance of approximately 12 * sqrt(2*6/9.81) * 4.42 ≈ 58.6m. This slight overshoot on flat ground provides a margin for the complex terrain, which may reduce the effective range per bounce depending on the slopes encountered.",
  "height": 6.0,
  "hori

Debug: Starting simulation. Target bounces: 3
Debug: Initial position: Vec2d(0.0, 6.0), velocity: Vec2d(12.0, 0.0)
Debug: Reached target bounces (3) at step 6202.
Debug: Simulation finished. Bounces recorded: 3. Duration: 0.03s
Debug: Final bounce locations (first three): [13.596000000000071, -4.982198651900522, 0.835601687291569]
Simulation Result: Bounces=[13.596000000000071, -4.982198651900522, 0.835601687291569], Target Bounce Dist=0.84m, Error=49.16m

SimLM Iteration 2/3


2025-04-29 22:32:18,068 - INFO - AFC remote call 1 is done.
2025-04-29 22:32:18,068 - INFO - GoogleClient model:gemini-2.5-flash-preview-04-17, response: ```json
{
  "critique": "The previous attempt resulted in the projectile going backwards after the first bounce, leading to a total distance far short of the target. This indicates the first bounce likely occurred on a steep uphill slope. The flat ground approximation was not useful. To achieve the target distance, the first bounce needs to occur further out on potentially less steep terrain, and the horizontal velocity needs to be sufficient to maintain forward momentum.",
  "height": 15.0,
  "horizontal_velocity": 25.0
}
```
2025-04-29 22:32:18,363 - INFO - AFC is enabled with max remote calls: 10.


...

2025-04-29 23:14:05,197 - INFO - AFC remote call 1 is done.
2025-04-29 23:14:05,197 - INFO - GoogleClient model:gemini-1.5-flash, response: ```json
{
  "reasoning": "The ground is complex, making precise analytical calculation impossible.  A numerical approach is necessary.  I'll start with an initial guess based on average bounce distance of approximately 16.67 meters (50m / 3 bounces).  This will require iterative refinement.  The initial height should be sufficient to allow for at least three bounces.  The sinusoidal component of the ground will affect the bounce locations, so I'll start with a slightly higher initial velocity to compensate for potential losses in horizontal distance due to the uneven terrain.  I'll adjust the height and velocity iteratively using a numerical simulation.",
  "height": 10.0,
  "horizontal_velocity": 20.0
}
```

2025-04-29 23:14:05,420 - INFO - AFC is enabled with max remote calls: 10.


Debug: Starting simulation. Target bounces: 3
Debug: Initial position: Vec2d(0.0, 10.0), velocity: Vec2d(20.0, 0.0)
Debug: Reached target bounces (3) at step 8449.
Debug: Simulation finished. Bounces recorded: 3. Duration: 0.04s
Debug: Final bounce locations (first three): [28.13999999999949, -14.842710767906492, -58.78720781813611]
Simulation Result: Bounces=[28.13999999999949, -14.842710767906492, -58.78720781813611], Target Bounce Dist=-58.79m, Error=108.79m

SimLM Iteration 2/7


2025-04-29 23:14:06,559 - INFO - AFC remote call 1 is done.
2025-04-29 23:14:06,559 - INFO - GoogleClient model:gemini-1.5-flash, response: ```json
{
  "critique": "The first attempt significantly undershot the target, indicating the initial horizontal velocity was too low.  The negative x-coordinates suggest the projectile is bouncing back in the negative x direction, possibly due to the complex ground shape. Increasing the horizontal velocity and potentially adjusting the initial height to account for the ground's undulations will be necessary.  A higher initial height might also help to clear the initial dips in the ground profile before the first bounce.",
  "height": 12.0,
  "horizontal_velocity": 25.0
}
```

2025-04-29 23:14:06,883 - INFO - AFC is enabled with max remote calls: 10.


Debug: Starting simulation. Target bounces: 3
Debug: Initial position: Vec2d(0.0, 12.0), velocity: Vec2d(25.0, 0.0)
Debug: Reached target bounces (3) at step 9604.
Debug: Simulation finished. Bounces recorded: 3. Duration: 0.04s
Debug: Final bounce locations (first three): [39.59999999999882, 111.10000001858896, 156.8211473999808]
Simulation Result: Bounces=[39.59999999999882, 111.10000001858896, 156.8211473999808], Target Bounce Dist=156.82m, Error=106.82m

SimLM Iteration 3/7


2025-04-29 23:14:12,172 - INFO - AFC is enabled with max remote calls: 10.
2025-04-29 23:14:17,458 - INFO - AFC is enabled with max remote calls: 10.
2025-04-29 23:14:17,752 - ERROR - An error occurred with Google Gemini: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-1.5-flash'}, 'quotaValue': '15'}]}, {'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, 

In [ ]:
import pandas as pd
from pathlib import Path

results_dir = Path("results")
result_file = results_dir / "experiment_results.jsonl"

result_df = pd.read_json(result_file, lines=True)
config_df = pd.json_normalize(result_df["config"])
result_df = pd.concat([result_df, config_df], axis=1)
result_df = result_df.drop(columns=["config"])
result_df